# Мультимодальный RAG-пайплайн: Результаты анализа

**Система:** Мультимодальная RAG для русско-казахских технических документов  
**Типы блоков:** header, text, list, table, no_text → chart, image  
**ИИ-модели:** CLIP, BLIP, SBERT, GPT API

---
## 1. Настройка среды

In [ ]:
import sys, os, logging
import warnings
warnings.filterwarnings('ignore')

# Корень проекта
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Логирование
logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s')
logger = logging.getLogger('notebook')

# Путь к PDF
PDF_PATH = 'data/input_pdfs/diploma_example.pdf'
print(f'PDF: {PDF_PATH}')
print(f'Exists: {os.path.exists(PDF_PATH)}')

In [ ]:
# Проверка зависимостей
print('=== Проверка зависимостей ===')

# Tesseract OCR
try:
    import pytesseract
    version = pytesseract.get_tesseract_version()
    print(f'Tesseract OCR: v{version}')
except Exception as e:
    print(f'Tesseract OCR: НЕ УСТАНОВЛЕН!')
    print(f'  Скачайте: https://github.com/UB-Mannheim/tesseract/wiki')
    print(f'  При установке выберите языки: Russian, Kazakh, English')
    print(f'  Добавьте в PATH: C:\\Program Files\\Tesseract-OCR')

# PyMuPDF
import fitz
print(f'PyMuPDF: v{fitz.version[0]}')

# transformers (CLIP, BLIP)
try:
    import transformers
    print(f'Transformers: v{transformers.__version__}')
except ImportError:
    print('Transformers: НЕ УСТАНОВЛЕН (pip install transformers)')

# sentence-transformers (SBERT)
try:
    import sentence_transformers
    print(f'Sentence-Transformers: v{sentence_transformers.__version__}')
except ImportError:
    print('Sentence-Transformers: НЕ УСТАНОВЛЕН')

# FAISS
try:
    import faiss
    print(f'FAISS: OK')
except ImportError:
    print('FAISS: НЕ УСТАНОВЛЕН (pip install faiss-cpu)')

print()
print(f'Документ: {os.path.basename(PDF_PATH)}')
print(f'Страниц: {len(fitz.open(PDF_PATH))}')

---
## 2. Этап 1 — Извлечение блоков из PDF

PyMuPDF извлекает текстовые блоки и изображения.  
`BlockClassifier` (правила, без ИИ) определяет тип: **header**, **text**, **list**, **table**, **no_text**.

In [ ]:
import fitz
from src.pipeline.page_analyzer import PageAnalyzer

doc = fitz.open(PDF_PATH)
analyzer = PageAnalyzer()

all_blocks = []
for page_num in range(len(doc)):
    page = doc[page_num]
    blocks = analyzer.analyze_page(page, page_num + 1)
    all_blocks.extend(blocks)
doc.close()

print(f'Всего страниц: {len(fitz.open(PDF_PATH))}')
print(f'Всего блоков: {len(all_blocks)}')
print()

# Подсчёт типов
from collections import Counter
type_counts = Counter(b.block_type for b in all_blocks)
for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {t:10s}: {c}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, ax = plt.subplots(figsize=(8, 4))
types = list(type_counts.keys())
counts = list(type_counts.values())
colors_map = {
    'header': '#4ECDC4', 'text': '#45B7D1', 'list': '#96CEB4',
    'table': '#FFEAA7', 'no_text': '#DFE6E9', 'chart': '#FF6B6B',
    'image': '#A29BFE'
}
bar_colors = [colors_map.get(t, '#95A5A6') for t in types]

bars = ax.barh(types, counts, color=bar_colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Количество блоков')
ax.set_title('Распределение типов блоков (Этап 1)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
os.makedirs('report/img', exist_ok=True)
plt.savefig('report/img/block_types.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: report/img/block_types.png')

In [ ]:
# Примеры каждого типа блока
print('=' * 60)
print('ПРИМЕРЫ БЛОКОВ ПО ТИПАМ')
print('=' * 60)

shown_types = set()
for block in all_blocks:
    if block.block_type not in shown_types and block.text:
        shown_types.add(block.block_type)
        text_preview = block.text[:120].replace('\n', ' ')
        if len(block.text) > 120:
            text_preview += '...'
        print(f'\n[{block.block_type:10s}] Стр. {block.page_num}:')
        print(f'  {text_preview}')

---
## 3. Этап 2 — Классификация изображений (CLIP)

Все блоки `no_text` — это изображения. CLIP (ИИ) смотрит на каждое и определяет:
- **chart** — графики, диаграммы, гистограммы (нужна транскрипция)
- **image** — фотографии, логотипы, схемы (краткое описание)

In [ ]:
from src.preprocessing.image_extractor import ImageExtractor
from src.preprocessing.image_captioner import ImageClassifier

# Извлечение изображений
extractor = ImageExtractor(min_width=50, min_height=50)
doc_name = os.path.splitext(os.path.basename(PDF_PATH))[0]
images_dir = os.path.join('data/images', doc_name)
extracted_images = extractor.extract_from_pdf(PDF_PATH, output_dir=images_dir)
print(f'Извлечено изображений: {len(extracted_images)}')

# CLIP классификация
classifier = ImageClassifier(use_clip=True)

if extracted_images:
    classifications = classifier.classify_batch(extracted_images)

    # Разделение
    chart_images = [
        (img, cls) for img, cls in zip(extracted_images, classifications)
        if cls['block_type'] == 'chart'
    ]
    photo_images = [
        (img, cls) for img, cls in zip(extracted_images, classifications)
        if cls['block_type'] == 'image'
    ]

    print(f'\nРезультаты CLIP:')
    print(f'  Charts (графики): {len(chart_images)}')
    print(f'  Images (фото):    {len(photo_images)}')
    print()
    for i, (img, cls) in enumerate(zip(extracted_images, classifications)):
        bt = cls['block_type']
        cs = cls['chart_score']
        ims = cls['image_score']
        conf = cls['confidence']
        print(f'  [{i+1}] Стр. {img.page_num+1}: {bt:5s}  '
              f'(chart={cs:.3f}, image={ims:.3f}, conf={conf:.3f})')
else:
    chart_images = []
    photo_images = []
    print('Изображения не найдены.')

---
## 4. Этап 3 — Транскрипция графиков (OCR + CLIP + BLIP + GPT)

Каждый **chart** проходит полный анализ:
1. **OCR** — читает текст с графика (подписи, числа)
2. **CLIP** — определяет подтип (столбчатая, линейная, круговая...)
3. **BLIP** — генерирует описание на английском
4. **GPT** — переводит и дополняет описание на русском

Результат: **транскрипция** — текстовое описание графика для RAG.

In [ ]:
from src.preprocessing.chart_analyzer import ChartAnalyzer
from src.preprocessing.ocr_processor import OCRProcessor

ocr = OCRProcessor()
chart_analyzer = ChartAnalyzer(
    ocr_processor=ocr,
    use_clip=True,
    use_blip=True,
    use_opencv=True,
)

chart_only_images = [img for img, cls in chart_images]
chart_results = []

if chart_only_images:
    chart_results = chart_analyzer.analyze_batch(chart_only_images)
    print(f'Проанализировано графиков: {len(chart_results)}')
    for i, r in enumerate(chart_results):
        print(f'\n--- График {i+1} (стр. {r.page_num}) ---')
        print(f'  Подтип (CLIP): {r.chart_subtype_ru} (conf={r.confidence:.2f})')
        blip = r.blip_caption
        print(f'  BLIP: {blip[:80]}...' if len(blip) > 80 else f'  BLIP: {blip}')
        ocr_t = r.ocr_text
        print(f'  OCR текст: {ocr_t[:100]}...' if len(ocr_t) > 100 else f'  OCR текст: {ocr_t}')
        gpt = r.gpt_description
        print(f'  GPT описание: {gpt[:150]}...' if len(gpt) > 150 else f'  GPT описание: {gpt}')
else:
    print('Графики не найдены в документе.')

---
## 5. Визуализация: Оригинал графика → Транскрипция

Наглядная демонстрация для научного руководителя:  
**слева** — исходное изображение, **справа** — результат анализа системы.

In [ ]:
import textwrap

if chart_results and chart_only_images:
    n_charts = min(len(chart_results), 6)

    for idx in range(n_charts):
        img_data = chart_only_images[idx]
        analysis = chart_results[idx]

        fig, (ax_img, ax_text) = plt.subplots(
            1, 2, figsize=(14, 5),
            gridspec_kw={'width_ratios': [1, 1.2]}
        )

        # Левая часть — оригинальное изображение
        ax_img.imshow(img_data.image)
        ax_img.set_title(
            f'Оригинал (стр. {analysis.page_num})',
            fontsize=13, fontweight='bold'
        )
        ax_img.axis('off')

        # Правая часть — транскрипция
        ax_text.axis('off')

        lines = []
        lines.append(f"Тип: {analysis.chart_subtype_ru}")
        lines.append(f"Уверенность CLIP: {analysis.confidence:.0%}")
        lines.append("")
        if analysis.blip_caption:
            lines.append(f"BLIP (EN): {analysis.blip_caption}")
            lines.append("")
        if analysis.gpt_description:
            wrapped = textwrap.fill(analysis.gpt_description, width=50)
            lines.append(f"Описание (RU):")
            lines.append(wrapped)
            lines.append("")
        if analysis.ocr_text:
            ocr_short = analysis.ocr_text[:150]
            if len(analysis.ocr_text) > 150:
                ocr_short += '...'
            lines.append(f"OCR: {ocr_short}")

        transcription = "\n".join(lines)

        ax_text.text(
            0.05, 0.95, transcription,
            transform=ax_text.transAxes,
            verticalalignment='top',
            fontsize=10, fontfamily='monospace',
            bbox=dict(
                boxstyle='round,pad=0.5',
                facecolor='#F8F9FA',
                edgecolor='#DEE2E6', alpha=0.9
            )
        )
        ax_text.set_title('Транскрипция', fontsize=13, fontweight='bold')

        plt.tight_layout()
        save_path = f'report/img/chart_recognition_{idx+1}.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Сохранено: {save_path}')
        print()
else:
    print('Нет графиков для визуализации.')

---
## 6. Markdown и Chunking

Все блоки (текст + графики + фото) собираются в Markdown → разбиваются на чанки.

In [ ]:
from src.preprocessing.markdown_builder import MarkdownBuilder
from src.rag.chunker import MarkdownChunker
from src.layout.block import Block

# Создаём Block-объекты для chart и image
chart_blocks = []
for img_data, analysis in zip(chart_only_images, chart_results):
    chart_blocks.append(Block(
        block_type='chart',
        bbox=img_data.bbox,
        page_num=img_data.page_num + 1,
        text=analysis.to_chunk_text(),
        metadata=analysis.to_dict(),
    ))

image_blocks = []
for img_data, cls in photo_images:
    image_blocks.append(Block(
        block_type='image',
        bbox=img_data.bbox,
        page_num=img_data.page_num + 1,
        metadata={'image_path': img_data.saved_path or '', 'ocr_text': ''},
    ))

final_blocks = all_blocks + chart_blocks + image_blocks
print(f'Итого блоков для Markdown: {len(final_blocks)}')
print(f'  Текстовых: {len(all_blocks)}')
print(f'  Графиков:  {len(chart_blocks)}')
print(f'  Фото:      {len(image_blocks)}')

# Markdown
builder = MarkdownBuilder()
source_name = os.path.basename(PDF_PATH)
markdown = builder.build(final_blocks, source_name)
print(f'\nMarkdown: {len(markdown)} символов')

# Сохранение
md_dir = os.path.join('data/indexes', doc_name)
os.makedirs(md_dir, exist_ok=True)
md_path = os.path.join(md_dir, source_name + '.md')
builder.save(markdown, md_path)

# Chunking
chunker = MarkdownChunker(
    max_chunk_size=1000, chunk_overlap=100, min_chunk_size=50
)
chunks = chunker.chunk_markdown(markdown, source=source_name)
print(f'Чанков: {len(chunks)}')

In [ ]:
# Пример чанка с транскрипцией графика (если есть)
chart_chunk_found = False
for c in chunks:
    if '[Тип:' in c.text or 'График' in c.text:
        print('=== Пример чанка с транскрипцией графика ===')
        print(c.text)
        print(f'\n(Размер: {len(c.text)} символов)')
        chart_chunk_found = True
        break

if not chart_chunk_found:
    print('Пример текстового чанка:')
    print(chunks[0].text[:500])

In [ ]:
sizes = [len(c.text) for c in chunks]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sizes, bins=20, color='#45B7D1', edgecolor='white', alpha=0.8)
ax.axvline(
    x=sum(sizes)/len(sizes), color='#FF6B6B', linestyle='--',
    label=f'Среднее: {sum(sizes)/len(sizes):.0f} символов'
)
ax.set_xlabel('Размер чанка (символы)')
ax.set_ylabel('Количество')
ax.set_title('Распределение размеров чанков')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('report/img/chunk_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Мин: {min(sizes)}, Макс: {max(sizes)}, Среднее: {sum(sizes)/len(sizes):.0f}')

---
## 7. Эмбеддинги (SBERT) и индексация (FAISS)

In [ ]:
from src.rag.embedder import MultimodalEmbedder
from src.rag.vector_store import FAISSVectorStore

embedder = MultimodalEmbedder(use_clip=False)
embeddings = embedder.embed_chunks(chunks)
print(f'Эмбеддинги: shape={embeddings.shape}')
print(f'Размерность: {embeddings.shape[1]}')

# FAISS
vector_store = FAISSVectorStore(dimension=embedder.dimension)
metadata_list = [chunk.to_dict() for chunk in chunks]
vector_store.add(embeddings, metadata_list)

# Сохранение
index_dir = os.path.join('data/indexes', doc_name)
os.makedirs(index_dir, exist_ok=True)
vector_store.save(index_dir)
print(f'Индекс сохранён: {index_dir}')
print(f'Чанков в индексе: {vector_store.size}')

In [ ]:
from sklearn.manifold import TSNE
import numpy as np

if embeddings.shape[0] > 5:
    perplexity = min(30, embeddings.shape[0] - 1)
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    coords = tsne.fit_transform(embeddings)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        coords[:, 0], coords[:, 1],
        c='#45B7D1', alpha=0.6, s=30, edgecolors='white'
    )
    ax.set_title('t-SNE визуализация эмбеддингов чанков')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig('report/img/tsne_embeddings.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Сохранено: report/img/tsne_embeddings.png')
else:
    print('Слишком мало чанков для t-SNE')

---
## 8. RAG: Вопрос-Ответ по документу

Задаём вопросы — система ищет релевантные чанки через FAISS и генерирует ответ через GPT.  
Включены вопросы как по тексту, так и **по графикам** для проверки транскрипции.

In [ ]:
from src.rag.retriever import MultimodalRetriever
from src.rag.generator import RAGGenerator

retriever = MultimodalRetriever(
    embedder=embedder,
    vector_store=vector_store,
    top_k=5,
)
generator = RAGGenerator(model_name='gpt-oss-20b')
print(f'LLM доступен: {generator.is_available()}')

def ask(question):
    """Задать вопрос RAG-системе."""
    print(f'\n{"=" * 60}')
    print(f'Вопрос: {question}')
    print(f'{"=" * 60}')

    # Retrieval
    results = retriever.retrieve(question, top_k=5)
    context_parts = []
    for i, r in enumerate(results, 1):
        chunk_type = r.get('type', 'text')
        label = 'график' if chunk_type == 'image_caption' else 'текст'
        context_parts.append(f'[Фрагмент {i}, {label}]:\n{r["text"]}')

    context = '\n\n---\n\n'.join(context_parts)

    # Generation
    result = generator.generate(question, context)

    print(f'\nОтвет ({result["model"]}):')
    print(result['answer'])

    print(f'\nИсточники:')
    for i, r in enumerate(results[:3], 1):
        score = r.get('score', 0)
        page = r.get('page', '?')
        preview = r.get('text', '')[:80]
        print(f'  [{i}] Стр. {page}, score={score:.4f}: {preview}...')

    return result

In [ ]:
# Вопрос 1: общий
q1 = ask('О чём этот документ?')

In [ ]:
# Вопрос 2: технологии
q2 = ask('Какие методы и технологии описаны в документе?')

In [ ]:
# Вопрос 3: данные на графиках (проверка транскрипции)
q3 = ask('Какие данные представлены на графиках и диаграммах в документе?')

In [ ]:
# Вопрос 4: тренды из графиков
q4 = ask('Какие тренды или изменения показаны на диаграммах?')

In [ ]:
# Вопрос 5: мультимодальный (текст + графики)
q5 = ask('Сравни данные из таблиц и графиков документа.')

---
## 9. Итоговая статистика

In [ ]:
print('=' * 50)
print('ИТОГОВАЯ СТАТИСТИКА')
print('=' * 50)
print(f'Документ:  {os.path.basename(PDF_PATH)}')
print(f'Страниц:   {len(fitz.open(PDF_PATH))}')
print()
print('Блоки (этап 1):')
for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {t:10s}: {c}')
print()
print('Изображения (этап 2):')
print(f'  Всего извлечено: {len(extracted_images)}')
print(f'  Charts:          {len(chart_images)}')
print(f'  Images:          {len(photo_images)}')
print()
print('Транскрипции (этап 3):')
print(f'  Графиков проанализировано: {len(chart_results)}')
print()
print('Индекс:')
print(f'  Чанков:      {len(chunks)}')
print(f'  Размерность: {embeddings.shape[1]}')
print(f'  Размер:      {vector_store.size}')